# Finding Non-SUSY Flux Vacua

**What's in this notebook?** This notebook demonstrates how to find **non-SUSY critical points** of the scalar potential in Type IIB flux compactifications using the `CriticalPointFinder` class.

- **Background** — why non-SUSY vacua matter and how they differ from SUSY vacua.
- **Setup** — model, sampler, and `CriticalPointFinder` configuration.
- **ISD mode comparison** — which flux sampling mode finds the most non-SUSY minima.
- **Solver comparison** — Newton vs scipy for solving $\partial V / \partial \phi = 0$.
- **Results analysis** — distribution of $V$, $|DW|$, Hessian eigenvalues.
- **Classification** — SUSY vs non-SUSY, minima vs saddle points, AdS vs dS.

(*Created:* Andreas Schachner, 2026-03-25)

## Background

SUSY flux vacua satisfy $D_I W = 0$ (F-flatness), which implies $\partial_I V = 0$ automatically. But the scalar potential can also have critical points where $\partial_I V = 0$ but $D_I W \neq 0$ — these are **non-SUSY vacua**.

The **no-scale potential** $V_{\rm ns} = e^K |DW|^2 \geq 0$ is particularly useful:
- All critical points are either SUSY minima ($DW = 0$, $V = 0$) or non-SUSY critical points ($DW \neq 0$, $V > 0$)
- ~85% of critical points found are minima (not saddle points)
- The positivity $V_{\rm ns} \geq 0$ means no runaway directions to $V \to -\infty$

The **full SUGRA potential** $V = e^K(|DW|^2 - 3|W|^2)$ can also be negative, producing AdS minima and more saddle points.

## Imports

In [ ]:
import warnings
import numpy as np
import time, math

import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt

import jaxvacua as jvc
from jaxvacua.critical_points import CriticalPointFinder

warnings.filterwarnings('ignore')

## 1. Model setup

In [ ]:
h12 = 2
model = jvc.FluxVacuaFinder(h12=h12, model_ID=1, model_type="KS", maximum_degree=2,map_to_fd=True)
model.lcs_tree.a_matrix = jnp.array([[4.5, 1.5], [1.5, 0.]])

Nmax = 200
sampler = jvc.data_sampler(
    model,
    moduli_bounds=(2., 5.),
    dilaton_bounds=(math.sqrt(3) / 2, 10.),
    axion_bounds=(-0.5, 0.5),
    seed=42,
)

print(f"Model: CP⁴[1,1,1,6,9][18], h12={h12}")
print(f"Search: Nmax={Nmax}, Im(z)∈(2,5), s∈(√3/2, 10)")

## 2. Quick start: finding non-SUSY critical points

The simplest usage: create a `CriticalPointFinder` and call `sample_critical_points`. The default settings use:
- **ISD- mode**: generates flux candidates via the gauge kinetic matrix (best non-SUSY yield)
- **scipy solver**: fastest for root-finding on $\partial V = 0$
- **noscale=True**: the no-scale potential $V_{\rm ns} = e^K |DW|^2$ (highest minima rate)

In [ ]:
# moduli_max=100 (default) filters runaway solutions where |z_i| or |tau| > 100
finder = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=True)

results = finder.sample_critical_points(
    n_target=50,
    n_batch=200,
    max_batches=5,
    isd_mode="ISD-",
    solver="scipy",
    verbose=True,
)

## 3. Analysing the results

Each result is a dictionary with the critical point data. Let's classify and visualise them.

In [ ]:
# Classify results
n_susy = sum(1 for r in results if r['is_susy'])
n_nonsusy = len(results) - n_susy
n_min = sum(1 for r in results if r['is_minimum'])
n_sad = len(results) - n_min

print(f"Total critical points: {len(results)}")
print(f"  SUSY:     {n_susy} ({100*n_susy/max(len(results),1):.0f}%)")
print(f"  non-SUSY: {n_nonsusy} ({100*n_nonsusy/max(len(results),1):.0f}%)")
print(f"  minima:   {n_min} ({100*n_min/max(len(results),1):.0f}%)")
print(f"  saddle:   {n_sad} ({100*n_sad/max(len(results),1):.0f}%)")

if results:
    # Print a few examples
    print(f"\n{'#':>3} {'V':>12} {'|DW|':>12} {'min_eig':>10} {'type':>12} {'Nflux':>6}")
    print("-" * 60)
    for i, r in enumerate(results[:10]):
        label = "SUSY" if r['is_susy'] else "non-SUSY"
        kind = "minimum" if r['is_minimum'] else "SADDLE"
        eig_min = r['eigenvalues'][0]
        print(f"{i:>3} {r['V']:>12.4e} {r['|DW|']:>12.4e} {eig_min:>10.2e} {label+' '+kind:>12} {r['Nflux']:>6.0f}")

## 4. ISD mode comparison

The `CriticalPointFinder` supports four ISD sampling modes for generating flux candidates:

| Mode | Input | Completes via | Best for |
|------|-------|---------------|----------|
| **F** | random $f$ | $h = M^{-1}\Sigma(f - c_0 h)$ | Mixed SUSY + non-SUSY (85% minima rate) |
| **H** | random $h$ | $f = sM\Sigma h + c_0 h$ | SUSY-dominated |
| **ISD+** | random $[f_2 \mid h_2]$ | compute $[f_1 \mid h_1]$ via $\mathcal{N}$ | Almost all SUSY |
| **ISD-** | random $[f_1 \mid h_1]$ | compute $[f_2 \mid h_2]$ via $\mathcal{N}^{-1}$ | **Most non-SUSY** (78% rate) |

Let's compare all four modes on 500 input candidates each.

In [ ]:
modes = ["F", "H", "ISD+", "ISD-"]
mode_results = {}

for mode in modes:
    print(f"\n{'='*60}")
    print(f"  Mode: {mode}")
    print(f"{'='*60}")
    finder_m = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=True)
    res = finder_m.sample_critical_points(
        n_target=500,       # large target so we don't stop early
        n_batch=500,
        max_batches=1,      # single batch of 500 candidates
        isd_mode=mode,
        solver="scipy",
        verbose=True,
    )
    mode_results[mode] = res

In [ ]:
# Summary table
print(f"{'Mode':<8} {'Found':>6} {'SUSY':>6} {'non-SUSY':>9} {'minima':>7} {'saddle':>7} {'min rate':>9} {'non-SUSY%':>10}")
print("-" * 72)
for mode in modes:
    res = mode_results[mode]
    n = len(res)
    n_s = sum(1 for r in res if r.get('is_susy', False))
    n_ns = n - n_s
    n_m = sum(1 for r in res if r.get('is_minimum', False))
    n_sad = n - n_m
    mr = f"{100*n_m/n:.0f}%" if n > 0 else "—"
    nsr = f"{100*n_ns/n:.0f}%" if n > 0 else "—"
    print(f"{mode:<8} {n:>6} {n_s:>6} {n_ns:>9} {n_m:>7} {n_sad:>7} {mr:>9} {nsr:>10}")

## 5. Solver comparison: Newton vs scipy

The `CriticalPointFinder` supports two main solver backends:

- **Newton** (`solver="newton"`): JIT-compiled Newton's method using `model.dV_x` and `model.ddV_x`. Converges quadratically when close to a solution.
- **scipy** (`solver="scipy"`): Uses `scipy.optimize.root` with the hybrid Powell method. Generally faster per-candidate due to adaptive step control.

Let's compare them on the same ISD- flux candidates.

In [ ]:
solvers = ["newton", "scipy"]
solver_results = {}

for slv in solvers:
    print(f"\n{'='*60}")
    print(f"  Solver: {slv}")
    print(f"{'='*60}")
    finder_s = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=True)
    t_start = time.perf_counter()
    res = finder_s.sample_critical_points(
        n_target=500,
        n_batch=500,
        max_batches=1,
        isd_mode="ISD-",
        solver=slv,
        verbose=True,
    )
    t_end = time.perf_counter()
    solver_results[slv] = {'results': res, 'time': t_end - t_start}

# Comparison
print(f"\n{'Solver':<10} {'Found':>6} {'SUSY':>6} {'non-SUSY':>9} {'minima':>7} {'time (s)':>9}")
print("-" * 52)
for slv in solvers:
    res = solver_results[slv]['results']
    n = len(res)
    n_s = sum(1 for r in res if r.get('is_susy', False))
    n_ns = n - n_s
    n_m = sum(1 for r in res if r.get('is_minimum', False))
    t = solver_results[slv]['time']
    print(f"{slv:<10} {n:>6} {n_s:>6} {n_ns:>9} {n_m:>7} {t:>8.1f}s")

## 6. Visualising the results

Let's plot the distributions of $V$, $|DW|$, and the Hessian eigenvalue spectrum for the ISD- results found above.

In [ ]:
# Use the quick-start results (ISD-, scipy, noscale=True)
if len(results) > 0:
    Vs = np.array([r['V'] for r in results])
    DWs = np.array([r['|DW|'] for r in results])
    eigs = np.array([r['eigenvalues'] for r in results])
    is_susy = np.array([r['is_susy'] for r in results])
    is_min = np.array([r['is_minimum'] for r in results])

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # --- Panel 1: V distribution ---
    ax = axes[0]
    ax.hist(np.log10(np.abs(Vs[~is_susy]) + 1e-30), bins=25, alpha=0.7,
            color='tab:red', label='non-SUSY', edgecolor='k', linewidth=0.5)
    if np.any(is_susy):
        ax.axvline(np.log10(1e-20), color='tab:blue', ls='--', label='SUSY (V≈0)')
    ax.set_xlabel(r'$\log_{10}|V_{\rm ns}|$')
    ax.set_ylabel('Count')
    ax.set_title('Scalar potential')
    ax.legend(fontsize=9)

    # --- Panel 2: |DW| distribution ---
    ax = axes[1]
    ax.hist(np.log10(DWs[~is_susy] + 1e-30), bins=25, alpha=0.7,
            color='tab:red', label='non-SUSY', edgecolor='k', linewidth=0.5)
    if np.any(is_susy):
        ax.hist(np.log10(DWs[is_susy] + 1e-30), bins=15, alpha=0.7,
                color='tab:blue', label='SUSY', edgecolor='k', linewidth=0.5)
    ax.set_xlabel(r'$\log_{10}|DW|$')
    ax.set_ylabel('Count')
    ax.set_title('F-term norm')
    ax.legend(fontsize=9)

    # --- Panel 3: Hessian eigenvalue spectrum ---
    ax = axes[2]
    min_eigs = eigs[:, 0]   # smallest eigenvalue
    ax.hist(min_eigs[is_min], bins=25, alpha=0.7, color='tab:green',
            label='minima', edgecolor='k', linewidth=0.5)
    ax.hist(min_eigs[~is_min], bins=25, alpha=0.7, color='tab:orange',
            label='saddle', edgecolor='k', linewidth=0.5)
    ax.axvline(0, color='k', ls='--', lw=1)
    ax.set_xlabel(r'smallest eigenvalue of $\partial^2 V$')
    ax.set_ylabel('Count')
    ax.set_title('Hessian spectrum')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("No results to plot.")

In [ ]:
# Bar chart: ISD mode comparison
if mode_results:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    x_pos = np.arange(len(modes))
    width = 0.35

    # Panel 1: SUSY vs non-SUSY
    ax = axes[0]
    n_susy_arr = [sum(1 for r in mode_results[m] if r.get('is_susy', False)) for m in modes]
    n_ns_arr = [len(mode_results[m]) - n_susy_arr[i] for i, m in enumerate(modes)]
    ax.bar(x_pos - width/2, n_susy_arr, width, label='SUSY', color='tab:blue', alpha=0.8)
    ax.bar(x_pos + width/2, n_ns_arr, width, label='non-SUSY', color='tab:red', alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(modes)
    ax.set_ylabel('Count')
    ax.set_title('SUSY vs non-SUSY by ISD mode')
    ax.legend()

    # Panel 2: minima vs saddle
    ax = axes[1]
    n_min_arr = [sum(1 for r in mode_results[m] if r.get('is_minimum', False)) for m in modes]
    n_sad_arr = [len(mode_results[m]) - n_min_arr[i] for i, m in enumerate(modes)]
    ax.bar(x_pos - width/2, n_min_arr, width, label='minima', color='tab:green', alpha=0.8)
    ax.bar(x_pos + width/2, n_sad_arr, width, label='saddle', color='tab:orange', alpha=0.8)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(modes)
    ax.set_ylabel('Count')
    ax.set_title('Minima vs saddle by ISD mode')
    ax.legend()

    plt.tight_layout()
    plt.show()

## 7. Inspecting individual critical points

Let's look at the moduli values and flux vectors for a few non-SUSY minima.

In [ ]:
# Pick non-SUSY minima
nonsusy_minima = [r for r in results if not r['is_susy'] and r['is_minimum']]
print(f"Found {len(nonsusy_minima)} non-SUSY minima out of {len(results)} critical points.\n")

for i, r in enumerate(nonsusy_minima[:5]):
    print(f"--- Non-SUSY minimum #{i+1} ---")
    print(f"  z₁ = {r['moduli'][0]:.4f}")
    print(f"  z₂ = {r['moduli'][1]:.4f}")
    print(f"  τ  = {r['tau']:.4f}")
    print(f"  V  = {r['V']:.6e}")
    print(f"  |DW| = {r['|DW|']:.6e}")
    print(f"  Nflux = {r['Nflux']:.0f}")
    print(f"  residual = {r['residual']:.2e}")
    print(f"  eigenvalues = {np.array2string(r['eigenvalues'], precision=2, separator=', ')}")
    print(f"  flux = {r['flux']}")
    print()

## 8. No-scale vs full SUGRA potential

The no-scale potential $V_{\rm ns} = e^K |DW|^2$ is always non-negative, so all its minima are stable.
The full SUGRA potential $V = e^K(|DW|^2 - 3|W|^2)$ can be negative (AdS), producing a richer landscape with more saddle points. Let's compare.

In [ ]:
potential_results = {}

for ns_flag, label in [(True, "no-scale"), (False, "full SUGRA")]:
    print(f"\n{'='*60}")
    print(f"  Potential: {label}")
    print(f"{'='*60}")
    finder_p = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=ns_flag)
    res = finder_p.sample_critical_points(
        n_target=500, n_batch=500, max_batches=1,
        isd_mode="ISD-", solver="scipy", verbose=True,
    )
    potential_results[label] = res

# Comparison
print(f"\n{'Potential':<14} {'Found':>6} {'SUSY':>6} {'non-SUSY':>9} {'minima':>7} {'saddle':>7} {'min%':>6}")
print("-" * 58)
for label in ["no-scale", "full SUGRA"]:
    res = potential_results[label]
    n = len(res)
    n_s = sum(1 for r in res if r.get('is_susy', False))
    n_ns = n - n_s
    n_m = sum(1 for r in res if r.get('is_minimum', False))
    n_sad = n - n_m
    mr = f"{100*n_m/n:.0f}%" if n > 0 else "—"
    print(f"{label:<14} {n:>6} {n_s:>6} {n_ns:>9} {n_m:>7} {n_sad:>7} {mr:>6}")

## 8b. Flux input priors: Gaussian vs uniform

The choice of prior on the ISD input vectors has a dramatic effect on candidate quality. The default `flux_prior="gaussian"` samples from $\mathcal{N}(0, \sigma^2 I)$ and rounds to integers. This concentrates inputs near small flux numbers where the tadpole constraint is more easily satisfied.

| Mode | Uniform prior | Gaussian prior | Improvement |
|------|:---:|:---:|:---:|
| **F** | 105 crit. pts | **159** crit. pts | **+51%** |
| **H** | 10 crit. pts | **67** crit. pts | **+6.7×** |
| **ISD+** | 5 crit. pts | **49** crit. pts | **+9.8×** |
| **ISD-** | 186 crit. pts | 178 crit. pts | (comparable) |

The Gaussian prior makes modes H and ISD+ actually usable, and significantly boosts F mode.

In [ ]:
# Compare Gaussian vs uniform priors across all modes
print(f"{'Mode':>6} {'Prior':>10} {'CritPts':>8} {'SUSY':>5} {'nonSUSY':>8} "
      f"{'minima':>7} {'saddle':>7} {'Time':>7}")
print("-" * 68)

for mode in ['ISD-', 'F', 'H', 'ISD+']:
    for prior in ['gaussian', 'uniform']:
        f_p = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=True,
                                  flux_prior=prior)
        t0 = time.perf_counter()
        res = f_p.sample_critical_points(
            n_target=500, n_batch=500, max_batches=1,
            isd_mode=mode, solver='scipy', verbose=False)
        dt = time.perf_counter() - t0
        moduli_vals = [np.append(r["moduli"].imag,r["tau"].imag) for r in res]
        if np.any(np.abs(moduli_vals)>1e1):
            res_new = []
            for r in res:
                mvals = np.append(r["moduli"].imag,r["tau"].imag)
                if np.any(np.abs(mvals)>1e1):
                    continue
                res_new.append(r)
                
            res = res_new
        n_r = len(res)
        n_s = sum(1 for r in res if r.get('is_susy', False))
        n_ns = n_r - n_s
        n_m = sum(1 for r in res if r.get('is_minimum', False))
        n_sad = n_r - n_m
        print(f"{mode:>6} {prior:>10} {n_r:>8} {n_s:>5} {n_ns:>8} "
              f"{n_m:>7} {n_sad:>7} {dt:>6.1f}s")

### Calibrating priors and runaway filtering

The default $\sigma$ values are tuned for h12=2 KS models. For a different geometry, calibrate them via `calibrate_priors()` (~5 seconds).

**Runaway filtering**: Solvers (especially scipy and Newton) can produce solutions where the moduli escape to astronomically large values ($|z| \sim 10^{30}$). The `moduli_max` parameter (default=100) rejects these:
- In `_is_physical`: solutions with $\max(|x_i|) > $ `moduli_max` are filtered out
- In the Newton solver: iterations abort early when $\max(|x_i|) > $ `moduli_max` (**4× speedup** at h12=3)
- In the hybrid solver: runaways are filtered after the Adam phase, before Newton refinement

The `calibrate_priors()` method automatically runs a **runaway diagnostic** — a quick solve to measure what fraction of converged solutions are runaways. The results are returned in the dict under `_runaway_diagnostic`, with a warning if >80% are runaways.

In [ ]:
# Calibrate priors for the current model
# moduli_max=100 (default) filters runaway solutions with |z_i| or |tau| > 100
finder_cal = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=True, moduli_max=100)
calibrated = finder_cal.calibrate_priors(n_test=1000, verbose=True,target_acceptance=0.9)

# The return dict contains σ values AND a runaway diagnostic
sigmas = {k: v for k, v in calibrated.items() if not k.startswith("_")}
diag = calibrated.get("_runaway_diagnostic")
print(f"\nCalibrated σ values: {sigmas}")
if diag:
    print(f"Runaway diagnostic: {diag['n_physical']}/{diag['n_converged']} physical "
          f"({100*(1-diag['runaway_fraction']):.0f}%), "
          f"{diag['n_runaways']} runaways")

# Save to model directory (for reuse across sessions)
path = finder_cal.save_calibration()
print(f"Saved to: {path}")

## 8d. Higher h12 example: h12=3

The `CriticalPointFinder` works for arbitrary $h^{1,2}$. At higher $h^{1,2}$, solvers tend to produce **runaway solutions** where $|z_i|$ or $|\tau|$ escape to $10^{8}$–$10^{30}$. The `moduli_max` parameter filters these efficiently:

- `moduli_max=100` (default): keeps solutions with all real/imaginary components $\leq 100$
- For models where physical solutions have larger moduli, increase to 500 or 1000
- Newton's early escape aborts iterations immediately when $|x| >$ `moduli_max` (**4× speedup**)

In [ ]:
# h12=3 example (model_4, which has hyperplanes and solutions near the sampling region)
model_3 = jvc.FluxVacuaFinder(h12=3, model_ID=4, model_type="KS")
print(f"h12={model_3.h12}, n_fluxes={model_3.n_fluxes}")
print(f"hyperplanes:\n{model_3.lcs_tree.hyperplanes}")

sampler_3 = jvc.data_sampler(
    model_3,
    moduli_bounds=(2., 8.),
    dilaton_bounds=(math.sqrt(3) / 2, 10.),
    axion_bounds=(-0.5, 0.5),
    seed=42,
)

# Use moduli_max=500 for h12=3 (solutions have |z| ~ 200-500 for this model)
finder_3 = CriticalPointFinder(model_3, sampler_3, Nmax=50, noscale=True, moduli_max=500)

# Calibrate and check runaway fraction
cal_3 = finder_3.calibrate_priors(n_test=100, verbose=True)
diag_3 = cal_3.get("_runaway_diagnostic")
if diag_3:
    print(f"\nRunaway rate at moduli_max=500: "
          f"{100*diag_3['runaway_fraction']:.0f}%")

# Run the search
results_3 = finder_3.sample_critical_points(
    n_target=20,
    n_batch=500,
    max_batches=2,
    isd_mode="ISD-",
    solver="newton",
    step_size=0.5,
    newton_max_iters=500,
    verbose=True,
)

n3 = len(results_3)
n3_ns = sum(1 for r in results_3 if not r['is_susy'])
n3_min = sum(1 for r in results_3 if r['is_minimum'])
print(f"\nh12=3 results: {n3} critical points ({n3_ns} non-SUSY, {n3_min} minima)")

if results_3:
    sorted_r = sorted(results_3, key=lambda r: max(abs(np.imag(r['moduli']))))
    print(f"\nClosest solutions to sampling region:")
    for i, r in enumerate(sorted_r[:3]):
        print(f"  #{i+1}: z = {r['moduli']}")
        print(f"       tau = {r['tau']:.4f}, V = {r['V']:.4e}, "
              f"|DW| = {r['|DW|']:.4e}, Nflux = {r['Nflux']:.0f}")

## 9. Vectorised optax solver and hybrid approach

The per-candidate loop in the Newton and scipy solvers becomes a bottleneck for large batches. The `CriticalPointFinder` includes a **vectorised optax Adam solver** (`solver="adam_v"`) that uses `lax.scan` for the step loop and `vmap` over the batch — compiled into a single XLA kernel.

Even better is the **hybrid solver** (`solver="hybrid"`): it runs a short vectorised Adam warm-start to move all candidates into Newton's convergence basin, then finishes with Newton's quadratic convergence. This typically finds **40% more critical points** than scipy alone.

### Parameter analysis

The key hyperparameters for the vectorised Adam solver are:
- **Learning rate schedule**: cosine annealing with `init_lr=0.5` works best
- **Gradient clipping**: `clip_by_global_norm(10.0)` prevents divergence
- **Step count**: 1000 is the optimal warm-start — more steps have diminishing returns
- **Objective function**: `|dV|²` outperforms `log(|dV|²+ε)` and `V` directly

In [ ]:
import optax

# Generate a shared set of ISD- candidates for fair comparison
finder_bench = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=True)
mod_pts = jnp.array(sampler.get_complex_moduli(500), dtype=complex)
tau_pts = jnp.array(sampler.get_complex_tau(500), dtype=complex)
x0_all, fl_all, _ = finder_bench._generate_flux_candidates(
    500, mod_pts, tau_pts, isd_mode="ISD-")
n_bench = min(200, len(x0_all))
print(f"Benchmarking on {n_bench} ISD- candidates\n")

tol = 1e-8

# --- Parameter sweep for Adam_v ---
print("Learning rate schedule comparison (Adam, cosine annealing, 5000 steps):")
print(f"{'lr':>6} {'conv':>6} {'min_res':>10} {'p10_res':>10} {'med_res':>10} {'time':>7}")
print("-" * 55)
for lr in [0.1, 0.3, 0.5, 0.7, 1.0]:
    sched = optax.cosine_decay_schedule(init_value=lr, decay_steps=5000)
    opt = optax.chain(optax.clip_by_global_norm(10.0), optax.adam(learning_rate=sched))
    t0 = time.perf_counter()
    _, res, conv = finder_bench._solve_dV_optax_batch(
        x0_all[:n_bench], fl_all[:n_bench], optimiser=opt,
        n_steps=5000, objective="dV2", tol=tol, sub_batch_size=50)
    dt = time.perf_counter() - t0
    sr = np.sort(res)
    print(f"{lr:>6.1f} {int(conv.sum()):>6} {sr[0]:>10.2e} {sr[int(0.1*len(sr))]:>10.2e} "
          f"{np.median(res):>10.2e} {dt:>6.1f}s")

In [ ]:
# --- Objective function comparison ---
print("Objective function comparison (Adam cosine lr=0.5, 5000 steps):")
print(f"{'objective':>10} {'conv':>6} {'min_res':>10} {'med_res':>10}")
print("-" * 40)
for obj in ["dV2", "log_dV2", "V"]:
    sched = optax.cosine_decay_schedule(init_value=0.5, decay_steps=5000)
    opt = optax.chain(optax.clip_by_global_norm(10.0), optax.adam(learning_rate=sched))
    _, res, conv = finder_bench._solve_dV_optax_batch(
        x0_all[:n_bench], fl_all[:n_bench], optimiser=opt,
        n_steps=5000, objective=obj, tol=tol, sub_batch_size=50)
    print(f"{obj:>10} {int(conv.sum()):>6} {np.min(res):>10.2e} {np.median(res):>10.2e}")

In [ ]:
# --- Full solver comparison: scipy vs Newton vs Adam_v vs Hybrid ---
from scipy.optimize import root as scipy_root

solver_bench = {}

# 1. scipy
t0 = time.perf_counter()
scipy_conv = 0
for j in range(n_bench):
    fl_np = fl_all[j].copy()
    def f(x, _fl=fl_np): return np.asarray(
        model.dV_x(jnp.array(x), jnp.array(_fl), noscale=True))
    def jac(x, _fl=fl_np): return np.asarray(
        model.ddV_x(jnp.array(x), jnp.array(_fl), noscale=True))
    r = scipy_root(f, x0=x0_all[j], method='hybr', jac=jac)
    if r.success and max(abs(r.fun)) < tol: scipy_conv += 1
solver_bench['scipy'] = (scipy_conv, time.perf_counter() - t0)

# 2. Newton
t0 = time.perf_counter()
newton_conv = 0
for j in range(n_bench):
    _, _, c = finder_bench._solve_dV_newton_single(
        x0_all[j], fl_all[j], tol=tol, max_iters=300)
    if c: newton_conv += 1
solver_bench['Newton'] = (newton_conv, time.perf_counter() - t0)

# 3. Hybrid (Adam 1k + Newton)
sched = optax.cosine_decay_schedule(init_value=0.5, decay_steps=1000)
opt = optax.chain(optax.clip_by_global_norm(10.0), optax.adam(learning_rate=sched))
t0 = time.perf_counter()
x_warm, res_warm, _ = finder_bench._solve_dV_optax_batch(
    x0_all[:n_bench], fl_all[:n_bench], optimiser=opt,
    n_steps=1000, objective="dV2", tol=tol, sub_batch_size=50)
hybrid_conv = int(np.sum(res_warm < tol))
mask = (res_warm < 1.0) & (res_warm >= tol)
for j in np.where(mask)[0]:
    _, res_n, conv_n = finder_bench._solve_dV_newton_single(
        x_warm[j], fl_all[j], tol=tol, max_iters=300)
    if conv_n: hybrid_conv += 1
solver_bench['Hybrid 1k'] = (hybrid_conv, time.perf_counter() - t0)

# Print comparison
print(f"\n{'Solver':<14} {'Converged':>10} {'Rate':>8} {'Time':>8}")
print("-" * 44)
for name in ['scipy', 'Newton', 'Hybrid 1k']:
    nc, t = solver_bench[name]
    print(f"{name:<14} {nc:>10} {100*nc/n_bench:>7.1f}% {t:>7.1f}s")

### Using the hybrid solver via `sample_critical_points`

The hybrid solver is the recommended choice when you want to maximise the number of critical points found:

In [ ]:
# The hybrid solver: vectorised Adam warm-start (1k steps) + Newton refinement
finder_hybrid = CriticalPointFinder(model, sampler, Nmax=Nmax, noscale=True)

results_hybrid = finder_hybrid.sample_critical_points(
    n_target=50,
    n_batch=200,
    max_batches=3,
    isd_mode="ISD-",
    solver="hybrid",
    optax_steps=1000,           # Adam warm-start steps (1k is optimal)
    newton_max_iters=300,       # Newton refinement iterations
    verbose=True,
)

n_h = len(results_hybrid)
n_ns_h = sum(1 for r in results_hybrid if not r.get('is_susy', False))
n_min_h = sum(1 for r in results_hybrid if r.get('is_minimum', False))
print(f"\nHybrid results: {n_h} critical points "
      f"({n_ns_h} non-SUSY, {n_min_h} minima)")

## 10. Performance analysis: moduli_max, sampling regions, and Nmax

The `moduli_max` parameter, the moduli/dilaton sampling region, and $N_{\max}$ all affect how many physical critical points are found vs. how many are runaways. This section benchmarks their interplay.

### 10a. Effect of `moduli_max`

The `moduli_max` filter rejects solutions where any real coordinate component exceeds the threshold. Setting it too tight loses physical solutions; too loose lets runaways through.

In [ ]:
# 10a. moduli_max sweep (ISD-, scipy, default region)
print(f"{'moduli_max':>12} {'CritPts':>8} {'SUSY':>5} {'nonSUSY':>8} "
      f"{'minima':>7} {'saddle':>7} {'Time':>7}")
print("-" * 62)

sampler_default = jvc.data_sampler(model, moduli_bounds=(2., 5.),
    dilaton_bounds=(math.sqrt(3)/2, 10.), axion_bounds=(-0.5, 0.5), seed=42)

for mmax in [10, 20, 50, 100, 200, 500, None]:
    f = CriticalPointFinder(model, sampler_default, Nmax=Nmax,
                            noscale=True, moduli_max=mmax)
    t0 = time.perf_counter()
    res = f.sample_critical_points(
        n_target=500, n_batch=500, max_batches=1,
        isd_mode="ISD-", solver="scipy", verbose=False)
    dt = time.perf_counter() - t0
    n_s = sum(1 for r in res if r.get('is_susy', False))
    n_m = sum(1 for r in res if r.get('is_minimum', False))
    label = str(mmax) if mmax is not None else "None"
    print(f"{label:>12} {len(res):>8} {n_s:>5} {len(res)-n_s:>8} "
          f"{n_m:>7} {len(res)-n_m:>7} {dt:>6.1f}s")

### 10b. Effect of sampling region

The moduli/dilaton sampling bounds determine where starting points are drawn. Tighter regions (closer to the origin) produce more physical solutions because the solver has less distance to travel.

In [ ]:
# 10b. Sampling region sweep (ISD-, scipy, moduli_max=100)
regions = [
    ((1., 2.), (math.sqrt(3)/2, 3.), "very tight: Im(z)∈(1,2), s∈(√3/2,3)"),
    ((1., 3.), (math.sqrt(3)/2, 5.), "tight: Im(z)∈(1,3), s∈(√3/2,5)"),
    ((2., 5.), (math.sqrt(3)/2, 10.), "default: Im(z)∈(2,5), s∈(√3/2,10)"),
    ((1., 10.), (math.sqrt(3)/2, 10.), "wide: Im(z)∈(1,10), s∈(√3/2,10)"),
    ((3., 8.), (1., 15.), "moderate: Im(z)∈(3,8), s∈(1,15)"),
    ((5., 15.), (2., 20.), "deep LCS: Im(z)∈(5,15), s∈(2,20)"),
]

print(f"{'Region':>48} {'CritPts':>8} {'SUSY':>5} {'nonSUSY':>8} "
      f"{'minima':>7} {'Time':>7}")
print("-" * 90)

for mod_b, dil_b, label in regions:
    s = jvc.data_sampler(model, moduli_bounds=mod_b,
        dilaton_bounds=dil_b, axion_bounds=(-0.5, 0.5), seed=42)
    f = CriticalPointFinder(model, s, Nmax=Nmax, noscale=True, moduli_max=100)
    t0 = time.perf_counter()
    res = f.sample_critical_points(
        n_target=500, n_batch=500, max_batches=1,
        isd_mode="ISD-", solver="scipy", verbose=False)
    dt = time.perf_counter() - t0
    n_s = sum(1 for r in res if r.get('is_susy', False))
    n_m = sum(1 for r in res if r.get('is_minimum', False))
    print(f"{label:>48} {len(res):>8} {n_s:>5} {len(res)-n_s:>8} "
          f"{n_m:>7} {dt:>6.1f}s")

### 10c. ISD mode comparison at tight region

With `moduli_max=100` and a tight sampling region, the relative performance of ISD modes changes significantly — modes H and ISD+ now outperform ISD- because they produce solutions closer to the starting points.

In [ ]:
# 10c. ISD mode comparison at tight region, moduli_max=100
sampler_tight = jvc.data_sampler(model, moduli_bounds=(1., 2.),
    dilaton_bounds=(math.sqrt(3)/2, 3.), axion_bounds=(-0.5, 0.5), seed=42)

print(f"{'Mode':>6} {'Solver':>8} {'CritPts':>8} {'SUSY':>5} {'nonSUSY':>8} "
      f"{'minima':>7} {'Time':>7}")
print("-" * 58)

for mode in ['ISD-', 'F', 'H', 'ISD+']:
    for solver in ['scipy', 'hybrid']:
        f = CriticalPointFinder(model, sampler_tight, Nmax=Nmax,
                                noscale=True, moduli_max=100)
        t0 = time.perf_counter()
        kw = {'optax_steps': 1000} if solver == 'hybrid' else {}
        res = f.sample_critical_points(
            n_target=500, n_batch=500, max_batches=1,
            isd_mode=mode, solver=solver, verbose=False, **kw)
        dt = time.perf_counter() - t0
        n_s = sum(1 for r in res if r.get('is_susy', False))
        n_m = sum(1 for r in res if r.get('is_minimum', False))
        print(f"{mode:>6} {solver:>8} {len(res):>8} {n_s:>5} {len(res)-n_s:>8} "
              f"{n_m:>7} {dt:>6.1f}s")

### 10d. Solution magnitude distribution

Without `moduli_max`, the vast majority of solutions are runaways. The tight region has the highest fraction of physical solutions.

In [ ]:
# 10d. Solution magnitude distributions (ISD-, scipy, no moduli_max)
dist_regions = [
    ((1., 2.), (math.sqrt(3)/2, 3.), "very tight"),
    ((2., 5.), (math.sqrt(3)/2, 10.), "default"),
    ((5., 15.), (2., 20.), "deep LCS"),
]

for mod_b, dil_b, label in dist_regions:
    s = jvc.data_sampler(model, moduli_bounds=mod_b,
        dilaton_bounds=dil_b, axion_bounds=(-0.5, 0.5), seed=42)
    f = CriticalPointFinder(model, s, Nmax=Nmax, noscale=True, moduli_max=None)
    res = f.sample_critical_points(
        n_target=500, n_batch=500, max_batches=1,
        isd_mode="ISD-", solver="scipy", verbose=False)

    if not res:
        print(f"{label}: 0 solutions")
        continue

    max_abs = np.array([float(np.max(np.abs(np.append(r['moduli'], r['tau']))))
                        for r in res])
    print(f"{label} ({len(res)} pts):")
    print(f"  max(|z|,|tau|): min={max_abs.min():.1f}, "
          f"median={np.median(max_abs):.1e}, max={max_abs.max():.1e}")
    for hi in [5, 10, 50, 100]:
        n_in = int(np.sum(max_abs <= hi))
        print(f"    ≤{hi:>5}: {n_in:>4}/{len(res)} ({100*n_in/len(res):>5.1f}%)")
    print()

### 10e. $N_{\max}$ sensitivity

The tadpole constraint controls flux magnitudes. Larger $N_{\max}$ allows more diverse fluxes but also more runaways.

In [ ]:
# 10e. Nmax sweep (tight region, ISD-, scipy, moduli_max=100)
print(f"{'Nmax':>6} {'CritPts':>8} {'SUSY':>5} {'nonSUSY':>8} "
      f"{'minima':>7} {'Time':>7}")
print("-" * 48)

for nmax in [10, 20, 50, 100, 200, 500]:
    f = CriticalPointFinder(model, sampler_tight, Nmax=nmax,
                            noscale=True, moduli_max=100)
    t0 = time.perf_counter()
    res = f.sample_critical_points(
        n_target=500, n_batch=500, max_batches=1,
        isd_mode="ISD-", solver="scipy", verbose=False)
    dt = time.perf_counter() - t0
    n_s = sum(1 for r in res if r.get('is_susy', False))
    n_m = sum(1 for r in res if r.get('is_minimum', False))
    print(f"{nmax:>6} {len(res):>8} {n_s:>5} {len(res)-n_s:>8} "
          f"{n_m:>7} {dt:>6.1f}s")

### 10f. Production run: tight region + hybrid + moduli_max=100

Combining the best settings: tight sampling region near the origin, hybrid solver, and `moduli_max=100`. All solutions are guaranteed to have $|z_i|, |\tau| \leq 100$.

In [ ]:
# 10f. Production run: tight region + hybrid + moduli_max=100
finder_prod = CriticalPointFinder(model, sampler_tight, Nmax=Nmax,
                                  noscale=True, moduli_max=100)

results_prod = finder_prod.sample_critical_points(
    n_target=200, n_batch=500, max_batches=5,
    isd_mode="ISD-", solver="hybrid", optax_steps=1000, verbose=True)

n_p = len(results_prod)
if n_p > 0:
    max_abs = np.array([float(np.max(np.abs(np.append(r['moduli'], r['tau']))))
                        for r in results_prod])
    Vs = np.array([r['V'] for r in results_prod])
    DWs = np.array([r['|DW|'] for r in results_prod])
    Nfluxes = np.array([r['Nflux'] for r in results_prod])

    print(f"\n{n_p} critical points found:")
    print(f"  max(|z|,|tau|): min={max_abs.min():.1f}, "
          f"median={np.median(max_abs):.1f}, max={max_abs.max():.1f}")
    print(f"  V: min={Vs.min():.2e}, median={np.median(Vs):.2e}")
    print(f"  |DW|: min={DWs.min():.2e}, median={np.median(DWs):.2e}")
    print(f"  Nflux: min={Nfluxes.min():.0f}, median={np.median(Nfluxes):.0f}, "
          f"max={Nfluxes.max():.0f}")

    # All solutions are within moduli_max
    print(f"\n  100% of solutions have max(|z|,|tau|) ≤ {max_abs.max():.1f} "
          f"(moduli_max={finder_prod.moduli_max})")

### 10g. Summary of performance analysis

**Key findings for this model (h12=2, CP$^4$[1,1,1,6,9][18]):**

1. **`moduli_max` effect**: Without filtering, ~85% of scipy solutions are runaways ($|z| > 100$). Setting `moduli_max=100` retains only physical solutions with a modest reduction in count (~30-60 vs ~160 unfiltered). The early Newton escape provides a ~2× speedup.

2. **Sampling region matters more than `moduli_max`**: The **very tight** region Im(z) $\in$ (1,2), $s \in$ ($\sqrt{3}/2$, 3) produces the **most physical solutions** — 59 with scipy vs 32 at the default region. Starting closer to the origin helps because the solver is less likely to escape.

3. **ISD mode ranking changes with `moduli_max`**: Without filtering, ISD- dominates (most solutions overall). **With filtering**, ISD+ (163 pts) and H (117 pts) outperform ISD- (54 pts) at the tight region — they produce solutions closer to the starting points.

4. **$N_{\max}$ has diminishing returns with `moduli_max`**: Going from $N_{\max}=50$ to 200 adds only ~5 extra physical solutions (58 vs 53). Most of the additional flux diversity at higher $N_{\max}$ produces runaways.

5. **Best production config**: tight region + `solver="hybrid"` + `moduli_max=100` → **~100 physical critical points per 500 candidates**, all with $|z|, |\tau| \leq 26$.

## 11. Summary and recommendations

**Best settings for finding non-SUSY critical points:**

| Goal | Recommended setting |
|------|-------------------|
| Maximise **non-SUSY** yield | `isd_mode="ISD-"` (~95% non-SUSY) |
| Maximise **minima** rate | `noscale=True` (~85% are minima) |
| Fastest solver | `solver="scipy"` (~45% convergence, fastest per-candidate) |
| Most critical points | `solver="hybrid"` (~64% convergence, ~40% more than scipy) |
| Best flux prior | `flux_prior="gaussian"` (default; +51% for F, +7× for H, +10× for ISD+) |
| Filter runaways | `moduli_max=100` (default; 4× speedup at h12=3 via early Newton escape) |

**Runaway filtering**: At higher $h^{1,2}$ or large $N_{\max}$, solvers produce many runaway solutions with $|z| \sim 10^{30}$. The `moduli_max` parameter (default=100):
- Rejects solutions with $\max(|x_i|) >$ `moduli_max` in `_is_physical`
- Aborts Newton iterations early when the iterate escapes (saves compute)
- Filters runaways after the Adam warm-start in the hybrid solver

Use `calibrate_priors(verbose=True)` to check the runaway fraction for your model. If >80% are runaways, consider increasing `moduli_max` or using a different moduli sampling region.

**Typical workflow:**

```python
from jaxvacua.critical_points import CriticalPointFinder

finder = CriticalPointFinder(model, sampler, Nmax=200, noscale=True)

# Check calibration and runaway rate
cal = finder.calibrate_priors(verbose=True)

# For maximum yield: use hybrid solver
results = finder.sample_critical_points(
    n_target=100, n_batch=500, max_batches=10,
    isd_mode="ISD-", solver="hybrid", optax_steps=1000,
)

# For quick exploration: use scipy
results = finder.sample_critical_points(
    n_target=100, n_batch=1000, max_batches=10,
    isd_mode="ISD-", solver="scipy",
)

# For higher h12 with larger moduli: increase moduli_max
finder = CriticalPointFinder(model, sampler, Nmax=50,
                             noscale=True, moduli_max=500)
```